# Práctica 3 — Visión por Computador

1. **Tarea 1**: contar el dinero (monedas y billetes de euro) en una imagen y en vivo con la cámara.
2. **Tarea 2**: clasificar microplásticos (pellets, alquitrán y fragmentos) y evaluar con la imagen de test.

En ninguna de las dos tareas se usan modelos de aprendizaje profundo.

## Paquetes necesarios

In [ ]:
from collections import Counter, deque
from dataclasses import dataclass, field, replace

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import median_filter
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Tarea 1: contar dinero

**TAREA:** Los ejemplos ilustrativos anteriores permiten saber el número de monedas presentes en la imagen ideal proporcionada. ¿Cómo determinar la cantidad de dinero presente en ella? A partir únicamente de las dimensiones, una sugerencia implicaría en primer término identificar de forma interactiva (por ejemplo haciendo clic en la imagen) una moneda de un valor fijo en la imagen (por ejemplo de 1€) para permitir establecer una relación entre píxeles y milímetros. Tras obtener esa información y las dimensiones en milímetros de las distintas monedas, es factible calcular la cantidad de dinero en la imagen ideal.

Una vez resuelto el reto con la imagen ideal proporcionada, diseña un demostrador en vivo que a partir de la entrada de vídeo de la cámara, determine el dinero presente en la escena. Si bien no hay restricciones sobre utilizar medidas geométricas o de color, o cualquier clasificador, no deben hacer uso de modelos de aprendizaje profundo. ¿Funciona correctamente? ¿Se observan problemas?
Posibles extras serían la presencia de objetos que no son monedas y/o haya solape entre las monedas.

**IMPORTANTE:** No se admite el uso de modelos de aprendizaje profundo

**Nota:** Para establecer la correspondencia entre píxeles y milímetros, comentar que la moneda de un euro tiene un diámetro de 23.25 mm, la de 50 céntimos de 24.25, la de 20 céntimos de 22.25, etc.

## Planteamiento

**Monedas**

1. `detectar_monedas` / `localizar_hough` → círculos `(x, y, r)`.
2. `tipos_de_monedas` → `"bronce"`, `"oro"`, `"1e"` o `"2e"` por moneda (color en Lab).
3. `clasificar` → valor en céntimos por moneda a partir de las proporciones entre diámetros (`None` si es ambiguo).
4. `Seguimiento` → vota el valor de cada moneda entre análisis para que las etiquetas no parpadeen.
5. `dibujar_monedas` / `dibujar_resumen` → círculos, etiquetas y total sobre el frame.

**Extensión: billetes de la serie Europa** (`DETECTAR_BILLETES = True`; con `False` solo se buscan monedas)

1. **Segmentación** (`detectar_billetes`): diferencia de color con el folio + bordes débiles (Canny), cierre, relleno de las zonas pálidas del billete y apertura. Solo se busca dentro del folio, para que la mesa no se pegue a los billetes. Se aceptan los contornos grandes y rectangulares (admite billetes algo arrugados o doblados). Si dos billetes quedan unidos por una sombra, se intentan separar con aperturas mayores.
2. **Monedas falsas**: los círculos detectados dentro de un billete (números impresos, etc.) se descartan.
3. **Clasificación por umbrales de color** (`medir_color_billete`, `clasificar_billete`; tono en Lab relativo al color del folio, igual que en las monedas): gris → 5 €, rojo → 10 €, azul → 20 €, naranja → 50 €, verde → 100 €, amarillo-marrón → 200 €. Solo si el tono queda en la frontera 50/200 y hay monedas clasificadas, decide el tamaño (140 mm frente a 153 mm).

**Calibración**: los umbrales (`FRAC_COLOR_GRIS`, `FRAC_AZUL_MIN`, `FRAC_VERDE_MIN`, `TONO_ROJO_MAX`, `TONO_NARANJA_MAX`) están ajustados con billetes de 20 € y 50 €. Con `DEBUG_BILLETES = True` se imprimen las medidas de color de cada billete y de cada candidato, para ajustar el resto con billetes reales.

### Configuración

Monedas: modo de localización, frecuencia de análisis, diámetros oficiales y colores de dibujo.

In [ ]:
USAR_HOUGH = False           # False = segmentación por área; True = HoughCircles
USAR_COLOR_RELATIVO = True   # False = umbrales de tono absolutos (colores_moneda)
DEBUG = False

ANALIZAR_CADA = 10           # frames entre análisis
HISTORIA_VOTOS = 7           # análisis que recuerda cada moneda para votar su valor
MAX_PERDIDAS = 3             # análisis sin ver una moneda antes de olvidarla

# Diámetros oficiales en mm (clave = valor en céntimos)
DIAMETROS = {
    1: 16.25, 2: 18.75, 5: 21.25, 10: 19.75,
    20: 22.25, 50: 24.25, 100: 23.25, 200: 25.75,
}
SERIES = {"bronce": (1, 2, 5), "oro": (10, 20, 50)}
SALTO_GRUPO = {"bronce": 0.065, "oro": 0.045}
VALOR_BIMETAL = {"1e": 100, "2e": 200}

ETIQUETAS = {1: "1c", 2: "2c", 5: "5c", 10: "10c", 20: "20c", 50: "50c",
             100: "1EUR", 200: "2EUR",            # "€" no se dibuja con la fuente de OpenCV
             500: "5EUR", 1000: "10EUR", 2000: "20EUR", 5000: "50EUR",
             10000: "100EUR", 20000: "200EUR"}

COLOR_TIPO = {                                    # BGR
    "bronce": (40, 110, 210),
    "oro": (0, 210, 255),
    "1e": (255, 200, 0),
    "2e": (255, 100, 160),
}
FUENTE = cv2.FONT_HERSHEY_SIMPLEX

Billetes: tamaños, parámetros de segmentación y umbrales de color. La clave de los diccionarios es el valor en céntimos, igual que en las monedas.

In [ ]:
DETECTAR_BILLETES = True
DEBUG_BILLETES = False       # imprime las medidas de color de cada billete (para calibrar)

TAM_BILLETES = {             # (largo, alto) en mm
    500: (120, 62), 1000: (127, 67), 2000: (133, 72),
    5000: (140, 77), 10000: (147, 77), 20000: (153, 77),
}
COLOR_BILLETE = {            # BGR, solo para dibujar
    500: (150, 150, 150), 1000: (60, 60, 230), 2000: (230, 140, 40),
    5000: (0, 140, 255), 10000: (60, 190, 60), 20000: (40, 190, 230),
}

# Segmentación (fracciones relativas al tamaño de la imagen: no dependen de la resolución)
SEG_LADO_MAX = 960           # se trabaja con la imagen reducida a este lado mayor
CANNY_BILLETE = (8, 20)      # bordes débiles: el contorno del billete sobre el folio es suave
SIGMA_CANNY_BILLETE = 1.0    # suavizado previo a Canny (en la imagen reducida)
K_CIERRE_BILLETE = 0.02      # cierre para unir el contorno (fracción de la diagonal)
K_APERTURA_BILLETE = 0.03    # apertura para quitar ruido fino y restos pegados
AREA_MIN_BILLETE = 0.025     # área mínima (fracción de la imagen); las monedas quedan muy por debajo
HUECO_MAX_BILLETE = 0.35    # huecos mayores (fracción de la imagen) no se rellenan
K_SEPARAR_BILLETES = (0.05, 0.07, 0.09)  # aperturas para separar billetes unidos
VECES_MONEDA_BILLETE = 8      # un billete ocupa al menos 8 veces el área de una moneda (real >= 14)
RECT_MIN_BILLETE = 0.80      # área / área del rectángulo mínimo (arrugado o doblado < 1)
ASPECTO_BILLETE = (1.2, 2.4) # largo/alto. Real ~1.9; doblado puede bajar bastante

# Color. Tono = ángulo en el plano (a, b) de Lab respecto al color del papel, en grados:
#   rojo ~30-50, naranja ~60-75, amarillo/marrón ~80-95, verde ~120-200, azul ~200-300
C_COLOR = 8.0                # croma mínima para contar un píxel como "con color"
C_FUERTE = 12.0              # croma mínima para el tono cálido (ignora el fondo beige pálido)
FRAC_COLOR_GRIS = 0.45       # menos píxeles con color que esto -> 5 € (gris)
FRAC_AZUL_MIN = 0.20         # fracción azul (sobre los píxeles con color) -> 20 €
FRAC_VERDE_MIN = 0.25        # fracción verde (sobre los píxeles con color) -> 100 €
TONO_ROJO_MAX = 55.0         # tono cálido medio por debajo -> 10 €
TONO_NARANJA_MAX = 80.0      # entre rojo y esto -> 50 €; por encima -> 200 €
ZONA_DUDA_50_200 = 4.0       # +-grados alrededor de TONO_NARANJA_MAX donde se usa el tamaño

### Estructuras de datos

In [ ]:
@dataclass
class Moneda:
    x: int
    y: int
    r: float
    tipo: str
    valor: int | None = None          # céntimos; None = no se puede saber


@dataclass
class Billete:
    x: int
    y: int
    r: float                          # medio lado corto: escala para el seguimiento
    tipo: str                         # siempre "billete"
    valor: int | None = None          # céntimos
    caja: np.ndarray | None = None    # 4 esquinas (int) del rectángulo rotado
    largo_px: float = 0.0             # lado largo en píxeles

### Proporciones entre diámetros

Como la escala de la imagen es desconocida, el valor se deduce de la relación entre los radios de las monedas de una misma serie.

In [ ]:
def moneda_mas_cercana(proporcion, valor_ref, permitidas):
    """Moneda de `permitidas` cuyo ratio D_ref / D_moneda se parece más a `proporcion`."""
    return min(permitidas,
               key=lambda m: abs(DIAMETROS[valor_ref] / DIAMETROS[m] - proporcion))


def proporciones_de(valores):
    """Ratios grande/pequeña entre las monedas dadas. Clave (pequeña, grande)."""
    vs = sorted(valores, key=DIAMETROS.get)
    return {(a, b): DIAMETROS[b] / DIAMETROS[a]
            for i, a in enumerate(vs) for b in vs[i + 1:]}


def clasificar_dos_grupos(media_peq, media_gra, nombre):
    """Devuelve (valor del grupo pequeño, valor del grupo grande)."""
    prop = proporciones_de(SERIES[nombre])
    p = media_gra / media_peq
    par = min(prop, key=lambda k: abs(prop[k] - p))
    return par


def agrupar(indices, radios, salto):
    """Agrupa índices de monedas con radios parecidos. Grupos ordenados de menor a mayor."""
    if not indices:
        return []
    orden = sorted(indices, key=lambda i: radios[i])
    grupos = [[orden[0]]]
    for prev, i in zip(orden, orden[1:]):
        if radios[i] / radios[prev] - 1 > salto:
            grupos.append([i])
        else:
            grupos[-1].append(i)
    return grupos


def agrupar_max3(indices, radios, salto):
    """Como agrupar, pero sube el salto hasta que haya como mucho 3 grupos."""
    grupos = agrupar(indices, radios, salto)
    while len(grupos) > 3 and salto < 0.25:
        salto += 0.01
        grupos = agrupar(indices, radios, salto)
    return grupos

### Color de las monedas

Canal `a` de Lab para separar bronce de oro y diferencia centro–anillo para detectar las bimetálicas (1 € y 2 €).

In [ ]:
A_CORTE_DEF = 0.5       # canal 'a' de corte si todas las monedas parecen del mismo tipo
GAP_MIN_A = 3.0         # salto mínimo en 'a' para considerar que hay bronce Y oro
BIMETAL_DELTA = 7.0     # diferencia centro-anillo para considerar 1 € / 2 €


def _lab_y_distancia(crop_rgb):
    h, w = crop_rgb.shape[:2]
    yy, xx = np.ogrid[:h, :w]
    dist = np.hypot(yy - h / 2, xx - w / 2) / (min(h, w) / 2)
    img = cv2.GaussianBlur(crop_rgb, (5, 5), 0)
    lab = cv2.cvtColor(img.astype(np.float32) / 255, cv2.COLOR_RGB2LAB)
    return lab[..., 0], lab[..., 1], lab[..., 2], dist


def colores_moneda(crop_rgb):
    """Método original con umbrales absolutos.
    Devuelve (oro, plata, bronce): 0 no aparece, 1 en el centro, 2 solo en el anillo."""
    R_CENTRO, R_ANILLO = 0.45, (0.75, 0.92)
    MIN_FRAC, SAT_PLATA, TONO_CORTE = 0.15, 0.22, 65

    L, a, b, dist = _lab_y_distancia(crop_rgb)
    saturacion = np.hypot(a, b) / np.maximum(L, 1)
    tono = np.degrees(np.arctan2(b, a))

    validos = (L > 30) & (L < 95)
    plata = saturacion < SAT_PLATA
    oro = ~plata & (tono >= TONO_CORTE) & (tono < 115)
    bronce = ~plata & (tono < TONO_CORTE) & (tono > -20)

    def presentes(zona):
        zona = zona & validos
        cuentas = np.array([(oro & zona).sum(), (plata & zona).sum(), (bronce & zona).sum()])
        total = cuentas.sum()
        return cuentas / total >= MIN_FRAC if total else np.zeros(3, bool)

    en_centro = presentes(dist <= R_CENTRO)
    en_anillo = presentes((dist >= R_ANILLO[0]) & (dist <= R_ANILLO[1]))
    return tuple(1 if c else 2 if r else 0 for c, r in zip(en_centro, en_anillo))


def tipo_desde_colores(colores):
    oro, plata, bronce = colores
    if bronce == 1:
        return "bronce"
    if plata == 1 and oro == 2:
        return "1e"          # centro plata, anillo oro
    if oro == 1 and plata == 2:
        return "2e"          # centro oro, anillo plata
    return "oro"


def medir_color(crop_rgb):
    """Media (a, b) del centro y del anillo de la moneda."""
    L, a, b, dist = _lab_y_distancia(crop_rgb)
    validos = (L > 20) & (L < 97)

    def media(zona):
        z = zona & validos
        return np.array([a[z].mean(), b[z].mean()]) if z.sum() >= 20 else None

    return media(dist <= 0.45), media((dist >= 0.72) & (dist <= 0.90))


def tipos_de_monedas(crops):
    """Tipo de cada recorte. En modo relativo compara las monedas del frame entre sí
    (robusto al tinte de la cámara)."""
    if not USAR_COLOR_RELATIVO:
        return [tipo_desde_colores(colores_moneda(c)) for c in crops]

    medidas = [medir_color(c) for c in crops]
    a_centro = np.array([c[0] if c is not None else 0.0 for c, _ in medidas])

    corte = A_CORTE_DEF
    if len(a_centro) > 1:
        orden = np.sort(a_centro)
        saltos = np.diff(orden)
        i = int(np.argmax(saltos))
        if saltos[i] >= GAP_MIN_A:
            corte = (orden[i] + orden[i + 1]) / 2

    tipos = []
    for (centro, anillo), a in zip(medidas, a_centro):
        if a > corte and (anillo is None or anillo[0] > corte):
            tipos.append("bronce")
        elif (centro is not None and anillo is not None
              and np.hypot(*(centro - anillo)) > BIMETAL_DELTA):
            tipos.append("2e" if centro[1] > anillo[1] else "1e")
        else:
            tipos.append("oro")
    return tipos

### Localización de monedas

In [ ]:
def _fondo_mediana(lab_s, k, tapar=None):
    """Fondo por mediana de ventana grande. `tapar` = píxeles que no son folio: se
    rellenan antes (inpainting) para que no contaminen la mediana."""
    if tapar is not None and tapar.any():
        lab_s = cv2.inpaint(lab_s, tapar.astype(np.uint8) * 255, 3, cv2.INPAINT_TELEA)
    return np.stack([cv2.medianBlur(lab_s[..., i], k) for i in range(3)], -1).astype(np.float32)


def detectar_monedas(img_bgr, k_ruido=6.0, umbral_min=10.0, umbral_max=40.0):
    """Radio por área (subpíxel) con fondo estimado localmente.
    Sirve con degradados de luz y fondos de cualquier color. Supone monedas sin tocarse."""
    h, w = img_bgr.shape[:2]
    s = 4
    small = cv2.resize(img_bgr, (w // s, h // s), interpolation=cv2.INTER_AREA)
    lab_s = cv2.cvtColor(small, cv2.COLOR_BGR2LAB)
    k = int(max(h, w) / s * 0.28) | 1
    bg = cv2.resize(_fondo_mediana(lab_s, k), (w, h), interpolation=cv2.INTER_LINEAR)

    lab = cv2.cvtColor(cv2.GaussianBlur(img_bgr, (5, 5), 0), cv2.COLOR_BGR2LAB).astype(np.float32)
    dif = np.linalg.norm(lab - bg, axis=2)
    med = np.median(dif)
    mad = np.median(np.abs(dif - med))
    umbral = max(med + k_ruido * 1.4826 * mad, umbral_min)

    if umbral > umbral_max:
        # Umbral absurdo: los objetos (monedas muy juntas, billetes) ocupan tanto que la
        # mediana ya no es el folio. Se marcan como "no folio" los píxeles más oscuros o con
        # más color que el folio (que es lo más claro de su zona), se tapan y se rehace el
        # fondo; el ruido se mide solo sobre el folio.
        ls = lab_s.astype(np.float32)
        L_folio = cv2.blur(cv2.dilate(ls[..., 0], np.ones((k, k), np.uint8)), (k, k))
        claro = ls[..., 0] >= L_folio - 15
        a0, b0 = np.median(ls[..., 1][claro]), np.median(ls[..., 2][claro])
        d0 = np.sqrt(np.clip(L_folio - ls[..., 0], 0, None) ** 2
                     + (ls[..., 1] - a0) ** 2 + (ls[..., 2] - b0) ** 2)
        tapar = cv2.dilate((d0 > 25).astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)
        bg = cv2.resize(_fondo_mediana(lab_s, k, tapar), (w, h), interpolation=cv2.INTER_LINEAR)
        dif = np.linalg.norm(lab - bg, axis=2)
        folio = cv2.resize(tapar.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST) == 0
        if folio.any():
            med = np.median(dif[folio])
            mad = np.median(np.abs(dif[folio] - med))
        umbral = max(med + k_ruido * 1.4826 * mad, umbral_min)

    mask = (dif > umbral).astype(np.uint8) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))

    def circulo(c, dx=0, dy=0):
        hull = cv2.convexHull(c)
        area = cv2.contourArea(hull)
        per = cv2.arcLength(hull, True)
        if area < 0.001 * h * w or per == 0:
            return None
        if np.sqrt(area / np.pi) > 0.25 * min(h, w):  # demasiado grande para ser una moneda
            return None
        if 4 * np.pi * area / per ** 2 < 0.80:       # circularidad
            return None
        (_, _), (rw, rh), _ = cv2.minAreaRect(hull)
        if max(rw, rh) > 1.25 * min(rw, rh):          # alargado: dos monedas unidas
            return None
        M = cv2.moments(hull)
        return (int(M["m10"] / M["m00"]) + dx, int(M["m01"] / M["m00"]) + dy,
                float(np.sqrt(area / np.pi)))         # radio sin redondear: más precisión

    contornos, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    circulos = []
    for c in contornos:
        circ = circulo(c)
        if circ is not None:
            circulos.append(circ)
            continue
        # Monedas unidas por un "puente" fino (sombras, muy juntas): se abre con un disco
        # cada vez mayor; abrir un círculo con un disco más pequeño no cambia su radio
        x, y, bw, bh = cv2.boundingRect(c)
        if bw * bh > 0.04 * h * w or min(bw, bh) < 10:
            continue
        sola = np.zeros((bh, bw), np.uint8)
        cv2.drawContours(sola, [c - [x, y]], -1, 255, -1)
        for frac in (0.2, 0.3, 0.4, 0.5, 0.6):
            kk = int(frac * min(bw, bh)) | 1
            partes, _ = cv2.findContours(
                cv2.morphologyEx(sola, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kk, kk))),
                cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            if len(partes) < 2:
                continue
            for p in partes:                            # las redondas se quedan; el resto sigue
                r = circulo(p, x, y)
                if r is not None:
                    circulos.append(r)
                    cv2.drawContours(sola, [p], -1, 0, -1)
                    cv2.drawContours(sola, [p], -1, 0, kk)   # quita también su borde/puente
            if not sola.any():
                break
    return circulos


def localizar_hough(img_bgr):
    """HoughCircles + eliminación de solapados (se queda con el mayor)."""
    gris = cv2.medianBlur(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), 7)
    circ = cv2.HoughCircles(gris, cv2.HOUGH_GRADIENT_ALT, 1.5, 100,
                            param1=300, param2=0.85, minRadius=50, maxRadius=150)
    if circ is None:
        return []
    candidatos = sorted(np.round(circ[0]).astype(int).tolist(), key=lambda c: -c[2])
    elegidos = []
    for x, y, r in candidatos:
        if all(abs(x - ex) > max(r, er) or abs(y - ey) > max(r, er) for ex, ey, er in elegidos):
            elegidos.append((x, y, float(r)))
    return elegidos

### Billetes: segmentación

In [ ]:
def _referencia_papel(lab):
    """Color (a, b) típico del folio y máscara de los píxeles usados (claros y poco saturados)."""
    L, a, b = lab[..., 0], lab[..., 1], lab[..., 2]
    croma = np.hypot(a - 128, b - 128)
    papel = (L >= np.percentile(L, 50)) & (croma <= np.percentile(croma, 50))
    return float(np.median(a[papel])), float(np.median(b[papel])), papel


def rellenar_huecos(mask, dif, ruido_papel, area_max, area_siempre):
    """Rellena los huecos de la máscara que pertenecen a un billete (sus zonas pálidas).

    Un hueco se rellena si es pequeño (reflejos, ventanas transparentes) o si, en media, se
    distingue algo del folio. Los huecos de folio limpio (p. ej. el espacio entre dos billetes
    que han quedado unidos por un borde) y los enormes se dejan vacíos."""
    contornos, jer = cv2.findContours(mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    salida = mask.copy()
    if jer is None:
        return salida
    for i, (*_, padre) in enumerate(jer[0]):
        if padre < 0:
            continue
        area = cv2.contourArea(contornos[i])
        if area > area_max:
            continue
        hueco = np.zeros_like(mask)
        cv2.drawContours(hueco, contornos, i, 255, -1)
        if area <= area_siempre or cv2.mean(dif, mask=hueco)[0] > 2 * ruido_papel:
            cv2.drawContours(salida, contornos, i, 255, -1)
    return salida


def detectar_billetes(img_bgr, circulos=()):
    """Rectángulos de billete en la imagen. Devuelve (lista de minAreaRect, (a0, b0) del papel).

    `circulos` son las monedas candidatas (x, y, r): sirven para no confundir un grupo de
    monedas muy juntas con un billete (el billete es mucho mayor que una moneda y los
    círculos que caen dentro de él, dibujos impresos, ocupan poco de su superficie).

    Máscara = (diferencia de color con el folio) OR (bordes débiles); se cierra, se rellenan
    los huecos (zonas pálidas del billete) y se abre para quitar ruido. Se aceptan los
    contornos grandes con forma de rectángulo."""
    h0, w0 = img_bgr.shape[:2]
    f = min(1.0, SEG_LADO_MAX / max(h0, w0))
    img = cv2.resize(img_bgr, None, fx=f, fy=f, interpolation=cv2.INTER_AREA) if f < 1 else img_bgr
    h, w = img.shape[:2]
    diag = np.hypot(h, w)

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
    lab_s = cv2.GaussianBlur(lab, (0, 0), 3)
    L, a, b = lab_s[..., 0], lab_s[..., 1], lab_s[..., 2]

    # Luminosidad local del folio (máximo en una ventana amplia, suavizado)
    s = 4
    L_red = cv2.resize(L, (w // s, h // s), interpolation=cv2.INTER_AREA)
    k = int(max(h, w) / s / 8) | 1
    L_fondo = cv2.resize(cv2.blur(cv2.dilate(L_red, np.ones((k, k), np.uint8)), (k, k)), (w, h))

    a0, b0, papel = _referencia_papel(lab_s)
    dif = np.hypot(0.5 * np.clip(L_fondo - L, 0, None), np.hypot(a - a0, b - b0))

    # Umbral: Otsu, pero sin pasar de 3x el ruido típico del folio (y nunca < 6)
    d8 = np.clip(dif * 4, 0, 255).astype(np.uint8)
    t_otsu, _ = cv2.threshold(d8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    umbral = max(6.0, min(t_otsu / 4, 3 * float(np.percentile(dif[papel], 90))))
    mask = (dif > umbral).astype(np.uint8) * 255

    gris = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2LAB)[..., 0], (0, 0), SIGMA_CANNY_BILLETE)
    mask |= cv2.Canny(gris, *CANNY_BILLETE)

    # Solo se busca dentro del folio: envolvente convexa de la mayor zona "de papel".
    # Así la mesa u otros objetos del borde de la imagen no se pegan a los billetes.
    elipse = lambda frac: cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(frac * diag) | 1,) * 2)
    zona_papel = cv2.morphologyEx((dif <= umbral).astype(np.uint8), cv2.MORPH_OPEN, elipse(0.005))
    n, etiquetas, stats, _ = cv2.connectedComponentsWithStats(zona_papel)
    if n > 1:
        mayor = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        pts = cv2.findNonZero((etiquetas == mayor).astype(np.uint8))
        folio = np.zeros_like(mask)
        cv2.fillConvexPoly(folio, cv2.convexHull(pts), 255)
        mask &= cv2.erode(folio, elipse(0.01))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, elipse(K_CIERRE_BILLETE))
    ruido_papel = max(1.0, float(np.median(dif[papel])))
    mask = rellenar_huecos(mask, dif, ruido_papel, HUECO_MAX_BILLETE * h * w, 0.01 * h * w)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, elipse(K_APERTURA_BILLETE))

    circ = [(x * f, y * f, r * f) for x, y, r in circulos]
    # percentil 25: los círculos falsos (dibujos del billete) suelen ser los grandes
    area_moneda = float(np.percentile([np.pi * r * r for *_, r in circ], 25)) if circ else 0.0

    def es_billete(c, dx=0, dy=0):
        area = cv2.contourArea(c)
        (cx, cy), (rw, rh), ang = cv2.minAreaRect(c)
        cx, cy = cx + dx, cy + dy
        if area < max(AREA_MIN_BILLETE * h * w, VECES_MONEDA_BILLETE * area_moneda) or min(rw, rh) == 0:
            return None
        caja = cv2.boxPoints(((cx, cy), (rw, rh), ang)).astype(np.float32)
        cubierta = sum(np.pi * r * r for x, y, r in circ
                       if cv2.pointPolygonTest(caja, (float(x), float(y)), False) >= 0)
        if cubierta > 0.4 * area:                                       # son monedas juntas
            return None
        rect = area / (rw * rh)
        aspecto = max(rw, rh) / min(rw, rh)
        if DEBUG_BILLETES:
            print(f"candidato billete: area={area / (h * w):.3f} rect={rect:.2f} aspecto={aspecto:.2f}")
        if rect < RECT_MIN_BILLETE or not ASPECTO_BILLETE[0] <= aspecto <= ASPECTO_BILLETE[1]:
            return None
        return ((cx / f, cy / f), (rw / f, rh / f), ang)            # a coordenadas originales

    rects = []
    contornos, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contornos:
        if cv2.contourArea(c) < AREA_MIN_BILLETE * h * w:
            continue
        r = es_billete(c)
        if r is not None:
            rects.append(r)
            continue
        # Puede ser un billete pegado a otra cosa (otro billete, una sombra...) por un "puente":
        # se abre con elementos cada vez mayores hasta que se separe en partes rectangulares
        x, y, bw, bh = cv2.boundingRect(c)
        sola = np.zeros((bh, bw), np.uint8)                              # solo su recuadro: más rápido
        cv2.drawContours(sola, [c - [x, y]], -1, 255, -1)
        for frac in K_SEPARAR_BILLETES:
            partes, _ = cv2.findContours(cv2.morphologyEx(sola, cv2.MORPH_OPEN, elipse(frac)),
                                         cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            buenas = [r for r in (es_billete(p, x, y) for p in partes) if r is not None]
            if buenas:
                rects.extend(buenas)
                break
    return rects, (a0, b0)

### Billetes: color y clasificación

In [ ]:
def medir_color_billete(lab, rect, papel_ab):
    """Medidas de color del interior del billete, relativas al color del folio."""
    h, w = lab.shape[:2]
    (cx, cy), (rw, rh), ang = rect
    interior = np.zeros((h, w), np.uint8)
    caja = cv2.boxPoints(((cx, cy), (rw * 0.85, rh * 0.85), ang)).astype(np.int32)
    cv2.fillConvexPoly(interior, caja, 1)
    sel = interior.astype(bool)

    a = lab[..., 1][sel] - papel_ab[0]
    b = lab[..., 2][sel] - papel_ab[1]
    croma = np.hypot(a, b)
    tono = (np.degrees(np.arctan2(b, a)) + 360) % 360

    con_color = croma > C_COLOR
    n_color = max(1, int(con_color.sum()))
    azul = con_color & (tono >= 200) & (tono < 300)
    verde = con_color & (tono >= 120) & (tono < 200)

    calido = (croma > C_FUERTE) & ((tono < 110) | (tono > 340))
    tono_c = np.where(tono > 180, tono - 360, tono)                   # rojo continuo alrededor de 0
    tono_calido = float(np.average(tono_c[calido], weights=croma[calido])) if calido.any() else None

    return dict(frac_color=float(con_color.mean()),
                frac_azul=float(azul.sum() / n_color),
                frac_verde=float(verde.sum() / n_color),
                tono_calido=tono_calido)


def clasificar_billete(med, largo_px=0.0, mm_por_px=None):
    """Valor en céntimos a partir de las medidas de color (umbrales)."""
    if med["frac_color"] < FRAC_COLOR_GRIS:
        return 500                                     # gris
    if med["frac_azul"] >= FRAC_AZUL_MIN and med["frac_azul"] >= med["frac_verde"]:
        return 2000                                    # azul
    if med["frac_verde"] >= FRAC_VERDE_MIN:
        return 10000                                   # verde
    t = med["tono_calido"]
    if t is None:
        return 500
    if t < TONO_ROJO_MAX:
        return 1000                                    # rojo
    # Naranja (50) frente a amarillo-marrón (200): si el tono está en la zona de duda y hay
    # monedas que den la escala, decide el largo (140 mm frente a 153 mm)
    if abs(t - TONO_NARANJA_MAX) <= ZONA_DUDA_50_200 and mm_por_px and largo_px:
        corte = (TAM_BILLETES[5000][0] + TAM_BILLETES[20000][0]) / 2
        return 20000 if largo_px * mm_por_px > corte else 5000
    return 5000 if t < TONO_NARANJA_MAX else 20000


def dentro_de_billete(circulo, billetes_rect, solape_max=0.3):
    """True si el círculo (x, y, r) tiene su centro dentro de un billete o lo pisa mucho."""
    x, y, r = circulo
    for rect in billetes_rect:
        caja = cv2.boxPoints(rect).astype(np.float32)
        if cv2.pointPolygonTest(caja, (float(x), float(y)), False) >= 0:
            return True
        # solape aproximado: fracción de puntos del disco que caen en el billete
        ang = np.linspace(0, 2 * np.pi, 24, endpoint=False)
        pts = [(x + rr * np.cos(a), y + rr * np.sin(a)) for rr in (0.35 * r, 0.7 * r) for a in ang]
        dentro = sum(cv2.pointPolygonTest(caja, (float(px), float(py)), False) >= 0 for px, py in pts)
        if dentro > solape_max * len(pts):
            return True
    return False

### Clasificación de monedas

In [ ]:
def clasificar(radios, tipos):
    """Valor en céntimos de cada moneda (None si no se puede deducir)."""
    valores = [VALOR_BIMETAL.get(t) for t in tipos]
    media = lambda grupo: float(np.mean([radios[i] for i in grupo]))

    def asignar(grupo, valor):
        for i in grupo:
            valores[i] = valor

    # 1) Series con 2 o 3 tamaños distintos: se resuelven solas
    grupos = {
        nombre: agrupar_max3([i for i, t in enumerate(tipos) if t == nombre],
                             radios, SALTO_GRUPO[nombre])
        for nombre in SERIES
    }
    sueltos = {}
    for nombre, gs in grupos.items():
        if len(gs) == 3:
            for g, v in zip(gs, SERIES[nombre]):
                asignar(g, v)
        elif len(gs) == 2:
            a, b = clasificar_dos_grupos(media(gs[0]), media(gs[1]), nombre)
            asignar(gs[0], a)
            asignar(gs[1], b)
        elif len(gs) == 1:
            sueltos[nombre] = gs[0]

    # 2) Series con un solo tamaño: se comparan con una moneda ya conocida
    conocidos = [v for v in valores if v is not None]
    if conocidos:
        valor_ref = max(conocidos)
        r_ref = media([i for i, v in enumerate(valores) if v == valor_ref])
        for nombre, g in sueltos.items():
            asignar(g, moneda_mas_cercana(r_ref / media(g), valor_ref, SERIES[nombre]))

    # 3) Solo un tamaño de bronce y uno de oro: se comparan entre ellos
    elif len(sueltos) == 2:
        g_b, g_o = sueltos["bronce"], sueltos["oro"]
        p = media(g_o) / media(g_b)
        b, o = min(((b, o) for b in SERIES["bronce"] for o in SERIES["oro"]),
                   key=lambda par: abs(DIAMETROS[par[1]] / DIAMETROS[par[0]] - p))
        asignar(g_b, b)
        asignar(g_o, o)

    # 4) Todas iguales: no hay referencia, se quedan en None
    return valores

### Análisis de un frame

Combina la localización de monedas y billetes y la clasificación de ambos.

In [ ]:
def analizar_frame(img_bgr):
    """Detecta y clasifica las monedas (y billetes) de un frame.
    Devuelve una lista de Moneda (y Billete si DETECTAR_BILLETES)."""
    circulos = localizar_hough(img_bgr) if USAR_HOUGH else detectar_monedas(img_bgr)
    rects, papel_ab = detectar_billetes(img_bgr, circulos) if DETECTAR_BILLETES else ([], None)
    # Lo "redondo" dentro de (o montado sobre) un billete es parte del dibujo, no una moneda
    circulos = [c for c in circulos if not dentro_de_billete(c, rects)]

    monedas = analizar_monedas(img_bgr, circulos) if circulos else []
    return monedas + analizar_billetes(img_bgr, rects, papel_ab, monedas)


def analizar_billetes(img_bgr, rects, papel_ab, monedas):
    if not rects:
        return []
    # Escala mm/px a partir de las monedas ya clasificadas (solo para desempatar 50/200)
    escalas = [DIAMETROS[m.valor] / (2 * m.r) for m in monedas if m.valor is not None]
    mm_por_px = float(np.median(escalas)) if escalas else None

    lab = cv2.cvtColor(cv2.GaussianBlur(img_bgr, (5, 5), 0), cv2.COLOR_BGR2LAB).astype(np.float32)
    billetes = []
    for rect in rects:
        (cx, cy), (rw, rh), _ = rect
        med = medir_color_billete(lab, rect, papel_ab)
        valor = clasificar_billete(med, max(rw, rh), mm_por_px)
        b = Billete(int(cx), int(cy), min(rw, rh) / 2, "billete", valor,
                    cv2.boxPoints(rect).astype(np.int32), max(rw, rh))
        if DEBUG_BILLETES:
            print(ETIQUETAS[valor], {k: (round(v, 3) if v is not None else None) for k, v in med.items()})
        billetes.append(b)
    return billetes


def analizar_monedas(img_bgr, circulos):
    """Clasifica las monedas a partir de sus círculos (x, y, r)."""

    # Recortes centrados, con relleno por si la moneda toca el borde
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    M = int(max(r for *_, r in circulos)) + 2
    pad = cv2.copyMakeBorder(img_rgb, M, M, M, M, cv2.BORDER_REPLICATE)
    crops = []
    for x, y, r in circulos:
        ri = int(round(r))
        crops.append(pad[y + M - ri: y + M + ri, x + M - ri: x + M + ri])

    tipos = tipos_de_monedas(crops)
    valores = clasificar([r for *_, r in circulos], tipos)

    monedas = [Moneda(x, y, r, t, v) for (x, y, r), t, v in zip(circulos, tipos, valores)]
    if DEBUG:
        for m in monedas:
            print(m)
    return monedas

### Seguimiento entre análisis

Cada moneda detectada se asocia con la más cercana del análisis anterior y se vota su valor.

In [ ]:
@dataclass
class Pista:
    x: int
    y: int
    r: float
    votos: deque = field(default_factory=lambda: deque(maxlen=HISTORIA_VOTOS))
    perdidas: int = 0
    obj: object = None                # última Moneda / Billete visto (posición, caja...)

    def moneda(self):
        valor, tipo = Counter(self.votos).most_common(1)[0][0]
        return replace(self.obj, valor=valor, tipo=tipo)


class Seguimiento:
    """Asocia cada moneda detectada con la del análisis anterior más cercana y vota su valor,
    para que las etiquetas no parpadeen."""

    def __init__(self):
        self.pistas: list[Pista] = []

    def actualizar(self, monedas):
        libres = list(self.pistas)
        for m in monedas:
            mismas = [p for p in libres if type(p.obj) is type(m)]     # moneda con moneda, billete con billete
            cerca = min(mismas, key=lambda p: np.hypot(p.x - m.x, p.y - m.y), default=None)
            if cerca is not None and np.hypot(cerca.x - m.x, cerca.y - m.y) < 0.7 * max(cerca.r, m.r):
                libres.remove(cerca)
                cerca.x, cerca.y, cerca.r, cerca.perdidas = m.x, m.y, m.r, 0
                pista = cerca
            else:
                pista = Pista(m.x, m.y, m.r)
                self.pistas.append(pista)
            pista.obj = m
            pista.votos.append((m.valor, m.tipo))

        for p in libres:
            p.perdidas += 1
        self.pistas = [p for p in self.pistas if p.perdidas <= MAX_PERDIDAS]

    def monedas_visibles(self):
        return [p.moneda() for p in self.pistas if p.perdidas <= 1]

### Dibujo y cámara

In [ ]:
def texto_con_borde(img, texto, org, escala=0.7, color=(255, 255, 255), grosor=2):
    cv2.putText(img, texto, org, FUENTE, escala, (0, 0, 0), grosor + 3, cv2.LINE_AA)
    cv2.putText(img, texto, org, FUENTE, escala, color, grosor, cv2.LINE_AA)


def dibujar_monedas(img, monedas):
    for m in monedas:
        if isinstance(m, Billete):
            color = COLOR_BILLETE.get(m.valor, (255, 255, 255))
            cv2.polylines(img, [m.caja], True, color, 4, cv2.LINE_AA)
        else:
            color = COLOR_TIPO[m.tipo]
            centro, r = (m.x, m.y), int(round(m.r))
            cv2.circle(img, centro, r, color, 3, cv2.LINE_AA)

        etiqueta = ETIQUETAS.get(m.valor, "?")
        (tw, th), base = cv2.getTextSize(etiqueta, FUENTE, 0.8, 2)
        x0, y0 = m.x - tw // 2, m.y + th // 2
        cv2.rectangle(img, (x0 - 5, y0 - th - 5), (x0 + tw + 5, y0 + base + 2), color, -1)
        cv2.putText(img, etiqueta, (x0, y0), FUENTE, 0.8, (0, 0, 0), 2, cv2.LINE_AA)


def resumen(monedas):
    """Devuelve (línea de recuento, línea de total)."""
    if not monedas:
        return "", ("No se ha detectado dinero" if DETECTAR_BILLETES else "No se han detectado monedas")

    cuenta = Counter(m.valor for m in monedas if m.valor is not None)
    recuento = "  ".join(f"{n}x{ETIQUETAS[v]}" for v, n in sorted(cuenta.items(), reverse=True))

    dudosas = [m for m in monedas if m.valor is None]
    if dudosas:
        n = len(dudosas)
        serie = SERIES.get(Counter(m.tipo for m in dudosas).most_common(1)[0][0], SERIES["oro"])
        seguro = sum(v * c for v, c in cuenta.items())        # billetes y monedas conocidas
        opciones = " o ".join(f"{(seguro + n * v) / 100:.2f}" for v in serie)
        return (recuento or f"{n} monedas iguales",
                f"Todas del mismo tamano: no se puede saber el valor ({opciones} EUR)")

    total = sum(v * n for v, n in cuenta.items()) / 100
    return recuento, f"Total: {total:.2f} EUR"


def dibujar_resumen(img, monedas):
    recuento, total = resumen(monedas)
    texto_con_borde(img, recuento, (10, 30), color=(255, 0, 255))
    texto_con_borde(img, total, (10, 65), color=(255, 0, 255))


def abrir_camara(indice=0):
    vid = cv2.VideoCapture(indice, cv2.CAP_DSHOW)   # CAP_DSHOW (Windows) para que apliquen los controles
    vid.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
    vid.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
    vid.set(cv2.CAP_PROP_AUTOFOCUS, 1)
    vid.set(cv2.CAP_PROP_AUTO_EXPOSURE, 0.25)
    vid.set(cv2.CAP_PROP_EXPOSURE, -6)
    vid.set(cv2.CAP_PROP_AUTO_WB, 1)                # con 0 se quedaba el tinte azul
    return vid

## Prueba con una imagen

In [ ]:
img = cv2.imread('Billetes.png')
seg_captura = Seguimiento()
seg_captura.actualizar(analizar_frame(img))
monedas_cap = seg_captura.monedas_visibles()
dibujar_monedas(img, monedas_cap)
dibujar_resumen(img, monedas_cap)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(img)
plt.show()

## Demostrador en vivo

- **ESPACIO**: guarda el frame en `captura.png`, lo analiza y lo muestra.
- **TAB / V**: alterna entre la cámara y la última captura.
- **ESC / Q**: salir.

In [ ]:
vid = abrir_camara()
seguimiento = Seguimiento()
contador = 0
captura = None            # imagen de captura.png ya analizada y dibujada
mostrando = "camara"      # "camara" o "captura"

while True:
    ret, frame = vid.read()
    if not ret:
        if cv2.waitKey(20) & 0xFF in (27, ord("q")):
            break
        continue

    tecla = cv2.waitKey(1) & 0xFF    # una sola lectura de teclado por vuelta

    if tecla == 32:  # ESPACIO: guarda el frame en PNG, lee ese PNG y lo analiza
        cv2.imwrite("captura.png", frame)          # se sobrescribe en cada captura
        img = cv2.imread("captura.png")
        seg_captura = Seguimiento()                # seguimiento nuevo solo para esta captura
        seg_captura.actualizar(analizar_frame(img))
        monedas_cap = seg_captura.monedas_visibles()
        dibujar_monedas(img, monedas_cap)
        dibujar_resumen(img, monedas_cap)
        captura = img
        mostrando = "captura"

    elif tecla in (9, ord("v")) and captura is not None:   # TAB o V: alterna
        mostrando = "camara" if mostrando == "captura" else "captura"

    # la cámara en directo sigue analizándose aunque estés viendo la captura
    if contador % ANALIZAR_CADA == 0:
        seguimiento.actualizar(analizar_frame(frame))
    contador += 1

    if mostrando == "captura":
        vista = captura.copy()
        texto = "CAPTURA (TAB/V = camara)"
    else:
        monedas = seguimiento.monedas_visibles()
        dibujar_monedas(frame, monedas)
        dibujar_resumen(frame, monedas)
        vista = frame
        texto = "CAMARA (ESPACIO = capturar, TAB/V = ver captura)"

    cv2.putText(vista, texto, (10, vista.shape[0] - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    cv2.imshow("Cam", vista)

    if tecla in (27, ord("q")):
        break

vid.release()
cv2.destroyAllWindows()

# Tarea 2: clasificación de microplásticos

La segunda tarea plantea la clasificación de microplásticos en una imagen real. Dado que el mundo real es muy variado, las imágenes no siempre se capturan con unas condiciones de iluminación tan buenas o controladas. Ejemplo con aplicación de variantes de umbralizados ofrecidas por OpenCV sobre una imagen de muestras de microplásticos.

**TAREA:** La tarea consiste en extraer características (geométricas y/o visuales) de las tres imágenes completas de partida, y *aprender* patrones que permitan identificar las partículas en nuevas imágenes. Para ello se proporciona como imagen de test *MPs_test.jpg* y sus correspondientes anotaciones *MPs_test_bbs.csv* con la que deben obtener las métricas para su propuesta de clasificación de microplásticos, además de la matriz de confusión. La matriz de confusión permitirá mostrar para cada clase el número de muestras que se clasifican correctamente de dicha clase, y el número de muestras que se clasifican incorrectamente como perteneciente a una de las otras dos clases. De nuevo, no deben hacer uso de modelos de aprendizaje profundo ni para segmentar ni para clasificar.

En el trabajo [SMACC: A System for Microplastics Automatic Counting and Classification](https://doi.org/10.1109/ACCESS.2020.2970498), las características geométricas utilizadas fueron:

- Área en píxeles
- Perímetro en píxeles
- Compacidad (relación entre el cuadrado del perímetro y el área de la partícula)
- Relación del área de la partícula con la del contenedor
- Relación del ancho y el alto del contenedor
- Relación entre los ejes de la elipse ajustada
- Definido el centroide, relación entre las distancias menor y mayor al contorno

Si no se quedan satisfechos con la segmentación obtenida, es el mundo real, también en el README comento técnicas recientes de segmentación, que podrían despertar su curiosidad.

**IMPORTANTE:** No se admite el uso de modelos de aprendizaje profundo

### Utilidades de cajas

Una caja es `((x_min, y_min), (x_max, y_max))`, con bordes incluidos.

In [ ]:
def normalizar(caja):
    (x1, y1), (x2, y2) = caja
    return ((int(min(x1, x2)), int(min(y1, y2))), (int(max(x1, x2)), int(max(y1, y2))))

def area(caja):
    (x1, y1), (x2, y2) = normalizar(caja)
    return (x2 - x1 + 1) * (y2 - y1 + 1)

def area_interseccion(a, b):
    (ax1, ay1), (ax2, ay2) = normalizar(a)
    (bx1, by1), (bx2, by2) = normalizar(b)
    ancho = min(ax2, bx2) - max(ax1, bx1) + 1
    alto = min(ay2, by2) - max(ay1, by1) + 1
    return ancho * alto if ancho > 0 and alto > 0 else 0

def ampliar(caja, m, w=None, h=None):
    """Caja agrandada m píxeles por cada lado (recortada a la imagen si se da w, h)."""
    (x0, y0), (x1, y1) = caja
    x0, y0, x1, y1 = x0 - m, y0 - m, x1 + m, y1 + m
    if w is not None:
        x0, y0, x1, y1 = max(0, x0), max(0, y0), min(w, x1), min(h, y1)
    return ((int(x0), int(y0)), (int(x1), int(y1)))

def recortar(c, obstaculo):
    """Recorta c por el lado que menos área le quite para que deje de tocar obstaculo (None si no queda nada)."""
    (x0, y0), (x1, y1) = normalizar(c)
    (ox0, oy0), (ox1, oy1) = normalizar(obstaculo)
    opciones = [((x0, y0), (ox0 - 1, y1)), ((ox1 + 1, y0), (x1, y1)),
                ((x0, y0), (x1, oy0 - 1)), ((x0, oy1 + 1), (x1, y1))]
    validas = [o for o in opciones if o[1][0] > o[0][0] and o[1][1] > o[0][1]]
    return max(validas, key=area) if validas else None

def fusionar_cajas(cajas, min_solape=0.5):
    """
    Quita los solapes entre cajas de la misma clase:
      - solape grande (>= min_solape de la pequeña): trozos del mismo objeto -> se unen;
      - solo se rozan: objetos distintos -> se recorta la pequeña.
    """
    cajas = [normalizar(c) for c in cajas]
    i = 0
    while i < len(cajas):
        for j in range(i + 1, len(cajas)):
            a, b = cajas[i], cajas[j]
            inter = area_interseccion(a, b)
            if inter > 0 and inter >= min_solape * min(area(a), area(b)):
                cajas[i] = ((min(a[0][0], b[0][0]), min(a[0][1], b[0][1])),
                            (max(a[1][0], b[1][0]), max(a[1][1], b[1][1])))
                cajas.pop(j)
                i = -1          # la caja ha crecido: volver a empezar
                break
        i += 1

    resultado = []
    for c in sorted(cajas, key=area, reverse=True):
        for r in resultado:
            if c is not None and area_interseccion(c, r) > 0:
                c = recortar(c, r)
        if c is not None:
            resultado.append(c)
    return resultado

def separar_cajas(cajas, prioritarias, max_perdida=0.5):
    """Quita el solape de 'cajas' con 'prioritarias' recortándolas; descarta las que perderían demasiado."""
    resultado = []
    for c in cajas:
        c, a0 = normalizar(c), area(c)
        for o in prioritarias:
            if c is not None and area_interseccion(c, o) > 0:
                c = recortar(c, o)
        if c is not None and area(c) >= (1 - max_perdida) * a0:
            resultado.append(c)
    return resultado

def sin_solape(cajas, otras):
    """Descarta las cajas que tocan alguna de 'otras'."""
    return [normalizar(c) for c in cajas if not any(area_interseccion(c, o) > 0 for o in otras)]

### Utilidades de imagen

In [ ]:
# Criterio de alquitrán (cámara y escáner): mediana del objeto oscura respecto al fondo y casi sin color.
OSCURO = 0.70     # gris / gris del fondo
CROMA_TAR = 5.5   # distancia al gris en el plano (a, b) de Lab

def croma(lab):
    labf = lab.astype(np.float32)
    return np.hypot(labf[..., 1] - 128, labf[..., 2] - 128)

def oscuridad_relativa(img):
    """Gris de cada píxel dividido por el del fondo local: no depende de la exposición."""
    h, w = img.shape[:2]
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(gris, None, fx=0.1, fy=0.1, interpolation=cv2.INTER_AREA)
    fondo = cv2.resize(cv2.medianBlur(small, 21), (w, h), interpolation=cv2.INTER_LINEAR)
    return gris.astype(np.float32) / np.maximum(fondo.astype(np.float32), 1)

def es_alquitran(rel, cr):
    return np.median(rel) < OSCURO and np.median(cr) <= CROMA_TAR

def diferencia_fondo(lab, esc, ancho, sigma):
    """
    Distancia de color (Lab) de cada píxel al fondo. El fondo es una mediana alargada en
    horizontal, para seguir las franjas del escáner sin "comerse" los objetos.
    """
    h, w = lab.shape[:2]
    small = cv2.resize(lab, None, fx=esc, fy=esc, interpolation=cv2.INTER_AREA)
    fondo = cv2.resize(median_filter(small, size=(3, ancho, 1), mode='nearest'), (w, h),
                       interpolation=cv2.INTER_LINEAR)
    diff = np.linalg.norm(lab.astype(np.float32) - fondo.astype(np.float32), axis=2)
    return cv2.GaussianBlur(diff, (0, 0), sigma)

def componentes(mask, area_min=0):
    """Recorre los componentes conectados: (caja, área, máscara local booleana)."""
    n, etiquetas, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8))
    for i in range(1, n):
        x, y, w, h, a = stats[i]
        if a >= area_min:
            yield (int(x), int(y), int(w), int(h)), int(a), etiquetas[y:y + h, x:x + w] == i

def forma(comp):
    """Contorno exterior y medidas de forma de una máscara local, o None si es degenerada."""
    c = max(cv2.findContours(comp.astype(np.uint8), cv2.RETR_EXTERNAL,
                             cv2.CHAIN_APPROX_NONE)[0], key=cv2.contourArea)
    A, P = cv2.contourArea(c), cv2.arcLength(c, True)
    if A <= 0 or P <= 0 or len(c) < 5:
        return None
    (_, _), (e1, e2), _ = cv2.fitEllipse(c)
    return dict(contorno=c,
                redondez=4 * np.pi * A / P ** 2,                      # 1 = círculo
                solidez=A / cv2.contourArea(cv2.convexHull(c)),       # 1 = sin entrantes
                elipse=min(e1, e2) / max(e1, e2))                     # 1 = nada alargado

### Pellets (PEL)

Objetos redondos, del tamaño esperado y que no son alquitrán.

In [ ]:
def obtener_PEL(img, r_med=30, redondez_min=0.78, solidez_min=0.95, elipse_min=0.60, margen=6):
    h, w = img.shape[:2]
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    mask = diferencia_fondo(lab, esc=0.1, ancho=41, sigma=2) > 10
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    cr, rel = croma(lab), oscuridad_relativa(img)

    cajas = []
    for (x, y, bw, bh), a, comp in componentes(mask, area_min=np.pi * r_med ** 2):
        if a > 4 * np.pi * r_med ** 2:
            continue
        f = forma(comp)
        if (f is None or f['redondez'] < redondez_min or f['solidez'] < solidez_min
                or f['elipse'] < elipse_min):
            continue
        # Interior (sin el borde): si es alquitrán, no es pellet
        k = max(3, int(0.2 * min(bw, bh)) | 1)
        nucleo = cv2.erode(comp.astype(np.uint8), np.ones((k, k), np.uint8)).astype(bool)
        if nucleo.sum() < 20:
            nucleo = comp
        if f['elipse'] >= 0.35 and es_alquitran(rel[y:y + bh, x:x + bw][nucleo],
                                                cr[y:y + bh, x:x + bw][nucleo]):
            continue
        cajas.append(ampliar(((x, y), (x + bw, y + bh)), margen, w, h))
    return fusionar_cajas(cajas)

### Alquitrán (TAR)

Manchas oscuras y sin color con forma de «gota».

In [ ]:
def es_pieza_no_gota(f, elipse_min=0.35):
    """Palitos (muy alargados) o triángulos: plástico negro, no alquitrán."""
    return f is not None and (f['elipse'] < elipse_min or
                              len(cv2.approxPolyDP(f['contorno'], 0.04 * cv2.arcLength(f['contorno'], True), True)) <= 3)

def obtener_TAR(img, recortesPel=(), area_min=250):
    rel = oscuridad_relativa(img)
    cr = croma(cv2.cvtColor(img, cv2.COLOR_BGR2LAB))
    cajas = []
    for (x, y, w, h), a, comp in componentes(rel < OSCURO, area_min):
        if not es_alquitran(rel[y:y + h, x:x + w][comp], cr[y:y + h, x:x + w][comp]):
            continue
        if es_pieza_no_gota(forma(comp)):
            continue
        cajas.append(((x, y), (x + w, y + h)))
    return sin_solape(fusionar_cajas(cajas), recortesPel)

### Fragmentos (FRA)

Todo lo demás que se distingue del fondo.

In [ ]:
def cajas_de_mascara(mask, area_min, margen=4, borde=15, lado_max_borde=300, grosor_min=12):
    h, w = mask.shape[:2]
    cajas = []
    for (x, y, bw, bh), a, _ in componentes(mask, area_min):
        toca_borde = x <= borde or y <= borde or x + bw >= w - borde or y + bh >= h - borde
        if toca_borde and max(bw, bh) > lado_max_borde:   # sombra / viñeteado
            continue
        if min(bw, bh) < grosor_min:                      # canto del papel, rayas
            continue
        cajas.append(ampliar(((x, y), (x + bw, y + bh)), margen, w, h))
    return fusionar_cajas(cajas)

def obtener_FRA(img, recortesPel=(), recortesTar=(), area_min=400):
    """Devuelve (fragmentos, alquitranes corregidos)."""
    h, w = img.shape[:2]
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    mask = (diferencia_fondo(lab, esc=0.125, ancho=51, sigma=1.5) > 9).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

    # 1) Un "alquitrán" rodeado casi por completo de objeto es una mancha de un fragmento.
    #    Se mira un anillo alrededor de la caja, saltándose el halo del propio objeto.
    def es_mancha(t, halo=8, frac=0.6):
        (x0, y0), (x1, y1) = t
        ext = ampliar(t, halo + max(10, int(0.5 * max(x1 - x0, y1 - y0))), w - 1, h - 1)
        intr = ampliar(t, halo, w - 1, h - 1)
        trozo = lambda c: mask[c[0][1]:c[1][1] + 1, c[0][0]:c[1][0] + 1]
        anillo = trozo(ext).size - trozo(intr).size
        return anillo > 0 and (int(trozo(ext).sum()) - int(trozo(intr).sum())) / anillo >= frac
    recortesTar = [t for t in recortesTar if not es_mancha(t)]
    detectadas = list(recortesPel) + list(recortesTar)

    # 2) Piezas negras que no son alquitrán (palitos, triángulos): se aíslan como el alquitrán
    rel, cr = oscuridad_relativa(img), croma(lab)
    negras = []
    for (x, y, bw, bh), a, comp in componentes(rel < OSCURO, area_min):
        if es_alquitran(rel[y:y + bh, x:x + bw][comp], cr[y:y + bh, x:x + bw][comp]):
            caja = ampliar(((x, y), (x + bw, y + bh)), 4, w, h)
            if not any(area_interseccion(caja, r) > 0 for r in detectadas):
                negras.append(caja)

    # 3) Se borra de la máscara todo lo ya clasificado (con margen para el halo) y se buscan
    #    los fragmentos en lo que queda; los restos pequeños pegados a lo borrado son su halo.
    g = 6
    borradas = [ampliar(c, g, w - 1, h - 1) for c in detectadas + negras]
    for (x0, y0), (x1, y1) in borradas:
        mask[y0:y1 + 1, x0:x1 + 1] = 0
    es_resto = lambda f: any(area_interseccion(ampliar(f, 2 * g), b) > 0 and area(f) < 0.5 * area(b)
                             for b in borradas)
    cajas = [f for f in cajas_de_mascara(mask, area_min) if not es_resto(f)]
    cajas = fusionar_cajas(cajas + negras)

    # 4) Prioridad pellet > alquitrán > fragmento
    return separar_cajas(cajas, detectadas), recortesTar

### Análisis completo y dibujo

La escala se estima automáticamente a partir de los objetos redondos.

In [ ]:
R_REF = 30        # r_med con el que están calibrados todos los umbrales
R_PEL_REF = 42.5  # radio equivalente de un pellet con r_med = R_REF

def estimar_r_med(img):
    """Escala de la imagen a partir de los objetos redondos, compactos y no negros (sobre todo pellets)."""
    h, w = img.shape[:2]
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    e = 200 / max(h, w)
    small = cv2.resize(lab, None, fx=e, fy=e, interpolation=cv2.INTER_AREA)
    fondo = cv2.resize(cv2.medianBlur(small, 15), (w, h))
    d = np.linalg.norm(lab.astype(np.float32) - fondo.astype(np.float32), axis=2)
    d = cv2.GaussianBlur(d, (0, 0), max(1, w / 2300))
    rel = oscuridad_relativa(img)
    diag = np.hypot(h, w)

    redondos, todos = [], []
    for (x, y, bw, bh), a, comp in componentes(d > 12):
        if not (diag * 0.003) ** 2 < a < (diag * 0.05) ** 2:
            continue
        f = forma(comp)
        if f is None:
            continue
        r = np.sqrt(a / np.pi)
        todos.append(r)
        if (f['redondez'] >= 0.8 and f['solidez'] >= 0.93
                and np.median(rel[y:y + bh, x:x + bw][comp]) >= 0.6):
            redondos.append(r)
    r_est = np.median(redondos) if len(redondos) >= 5 else np.median(todos)
    return R_REF * r_est / R_PEL_REF

def analizar(img, r_med=None):
    """
    Pellets, alquitrán y fragmentos de una imagen BGR. Devuelve (PEL, TAR, FRA) en coordenadas
    de la imagen original. r_med=None -> se estima la escala automáticamente.
    """
    f = R_REF / (r_med or estimar_r_med(img))
    if abs(f - 1) <= 0.05:
        f = 1
    else:
        img = cv2.resize(img, None, fx=f, fy=f,
                         interpolation=cv2.INTER_CUBIC if f > 1 else cv2.INTER_AREA)
    P = obtener_PEL(img, r_med=R_REF)
    T = obtener_TAR(img, P)
    F, T = obtener_FRA(img, P, T)
    # ceil en la esquina mínima y floor en la máxima: lo que no se tocaba sigue sin tocarse
    a_original = lambda cajas: [((int(np.ceil(x0 / f)), int(np.ceil(y0 / f))),
                                 (int(np.floor(x1 / f)), int(np.floor(y1 / f))))
                                for (x0, y0), (x1, y1) in cajas]
    return a_original(P), a_original(T), a_original(F)

def pintar(img, recortesPel=(), recortesTar=(), recortesFRA=()):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for cajas, color in [(recortesPel, (0, 0, 255)), (recortesTar, (255, 0, 0)), (recortesFRA, (0, 255, 0))]:
        for p0, p1 in cajas:
            cv2.rectangle(img_rgb, p0, p1, color, 2)
    return img_rgb

### Evaluación frente a las anotaciones

In [ ]:
def obtener_resultados_reales(path):
    """Lee el CSV de anotaciones: [(tipo, ((x0, y0), (x1, y1))), ...] con coordenadas enteras."""
    datos = []
    with open(path, "r") as file:
        file.readline()                        # cabecera
        for line in file:
            line = line.strip()
            if not line:
                continue
            tipo, x0, y0, x1, y1 = line.split(",")
            x0, y0, x1, y1 = (int(float(v)) for v in (x0, y0, x1, y1))   # ¡como números, no texto!
            datos.append((tipo.strip(), ((x0, y0), (x1, y1))))
    return datos


def chequeo(real, calculado):
    """
    Empareja cada objeto real con la caja calculada que más solapa (cada caja se usa una vez).
    Devuelve (real_out, resultado): los reales sin pareja -> resultado "NON"; las cajas
    calculadas sin pareja (falsos positivos) se añaden al final con real "NON".
    """
    usados = set()
    real_out, resultado = [], []

    for tipo_r, caja_r in real:
        mejor, mejor_area = None, 0
        for j, (tipo_c, caja_c) in enumerate(calculado):
            if j in usados:
                continue
            inter = area_interseccion(caja_r, caja_c)
            if inter > mejor_area:
                mejor, mejor_area = j, inter
        real_out.append(tipo_r)
        if mejor is None:
            resultado.append("NON")
        else:
            usados.add(mejor)
            resultado.append(calculado[mejor][0])

    for j, (tipo_c, _) in enumerate(calculado):
        if j not in usados:
            real_out.append("NON")
            resultado.append(tipo_c)

    return real_out, resultado

## Resultados sobre `MPs_test.jpg`

In [ ]:
img = cv2.imread("MPs_test.jpg")
recortesPel, recortesTar, recortesFRA = analizar(img)   # escala estimada automáticamente
img_out = pintar(img, recortesPel, recortesTar, recortesFRA)

calculado = ([("PEL", c) for c in recortesPel] +
             [("TAR", c) for c in recortesTar] +
             [("FRA", c) for c in recortesFRA])
real, resultado = chequeo(obtener_resultados_reales("MPs_test_bbs.csv"), calculado)

# Métricas (para más de una clase hay que indicar cómo promediar)
accuracy = accuracy_score(real, resultado)
precision = precision_score(real, resultado, average="weighted")
recall = recall_score(real, resultado, average="weighted")
f1score = f1_score(real, resultado, average="weighted")

print(f"Accuracy (TP/(n))= {accuracy:.3f}")
print(f"Precision (TP/(TP+FP)) = {precision:.3f}")
print(f"Recall (TP/(TP+FN)) = {recall:.3f}")
print(f"F1 Score (2*(precision*recall)/(precision+recall)) = {f1score:.3f}")

Matriz de confusión (`NON` = objeto anotado sin detectar o detección sin anotación).

In [ ]:
etiquetas = ["PEL", "TAR", "FRA", "NON"]
conf_matrix = confusion_matrix(real, resultado, labels=etiquetas)

with sns.plotting_context(font_scale=3.0):   # fuente grande solo para esta figura
    plt.figure(figsize=(8, 8))
    ax = sns.heatmap(conf_matrix, annot=True, fmt="d", cbar=False, cmap="flag",
                     xticklabels=etiquetas, yticklabels=etiquetas)
    ax.set_xlabel("Predicha", labelpad=-0.75, fontsize=25)
    ax.set_ylabel("Real/Anotado", labelpad=20, fontsize=25)
    ax.set_yticklabels(etiquetas, rotation=0)
    plt.show()

Detecciones sobre la imagen: azul = pellet, rojo = alquitrán, verde = fragmento.

In [ ]:
plt.figure(figsize=(14, 10))
plt.imshow(img_out)
plt.axis("off")
plt.show()